# DFL 전체 25fps · 25 epoch · 최소 검증 loss

GPU 런타임을 선택하고 **모두 실행**하세요. 기존 Drive의 `transition_final/transition_colab_data.zip`을 재사용합니다. 데이터 재업로드는 필요 없습니다.

- 5학습 / 1검증(J03WQQ) / 1평가(J03WR9), 전후반 분할 유지
- 선수+볼, 25fps 전체 인플레이, 4초(100프레임) 창
- 128차원 / 4헤드 / 블록 2, 총 배치 64, AMP 활성화
- **25 epoch 전부 실행, 얼리스탑 없음, 최소 검증 loss의 best.pt 선택**
- 5fps H/A, 경합 게이트, 5fps 경계 고정은 사용하지 않습니다. 기존 D 판정만 재사용합니다.
- 새 결과는 Drive `transition_final/runs/dfl_25fps_25epochs_minloss/`에 저장합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, zipfile, subprocess, csv, base64, gzip

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/transition_final')
archive = drive_root / 'transition_colab_data.zip'
old_project = Path('/content/transition_project')
project = old_project if (old_project / 'data/transition_final/source/J03WR9.parquet').is_file() else Path('/content/transition_25fps_project')
project.mkdir(parents=True, exist_ok=True)
if not (project / 'data/transition_final/source/J03WR9.parquet').is_file():
    if not archive.is_file():
        raise FileNotFoundError(f'기존 입력 ZIP 위치: {archive}')
    with zipfile.ZipFile(archive) as z:
        z.extractall(project)
os.chdir(project)
print('데이터 준비:', project)


In [ ]:
# 이 노트북에 최신 코드가 내장되어 있습니다. ZIP 속 예전 학습 코드를 별도로 고칠 필요 없습니다.
embedded = '...'
for relative, source in json.loads(gzip.decompress(base64.b64decode(embedded))).items():
    target = project / 'transition final' / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
print('전체 25fps 실행 코드 적용 완료')


In [ ]:
%pip -q install 'kloppy==3.18.0' pyarrow scipy pandas matplotlib tqdm scikit-learn torch-geometric


In [ ]:
RUN_NAME = 'dfl_25fps_25epochs_minloss'
MAX_EPOCHS = 25
MICROBATCH = 64  # CUDA 메모리 부족 시, 학습 시작 전에 32로 변경. 총 배치 64 유지.
EPOCH_WINDOW_CAP = 0  # 0 = 전체 인플레이 학습 창
RUN_BENCHMARK = False  # 필요할 때만 True: 2배치 실측 + 전체 검증

import torch
if not torch.cuda.is_available():
    raise RuntimeError('런타임 유형을 GPU로 변경하세요.')
print('GPU:', torch.cuda.get_device_name(0), '| AMP FP16: ON')
print('25fps 전체 인플레이 /', MAX_EPOCHS, 'epoch / 얼리스탑 OFF / 최소 검증 loss 선택')
run_dir = drive_root / 'runs' / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
settings = {'experiment':'full_25fps', 'epochs':MAX_EPOCHS, 'early_stop':0,
            'selection':'minimum_validation_loss', 'amp':True, 'amp_dtype':'float16',
            'microbatch':MICROBATCH, 'epoch_window_cap':EPOCH_WINDOW_CAP}
config_path = run_dir / 'colab_settings.json'
if config_path.exists() and json.loads(config_path.read_text()) != settings:
    raise ValueError('기존 실행과 설정이 다릅니다. RUN_NAME을 새 이름으로 바꾸세요.')
config_path.write_text(json.dumps(settings,ensure_ascii=False,indent=2))
with (project / 'transition final/parameters_25fps.csv').open(newline='',encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
changes = {'max_epochs':str(MAX_EPOCHS), 'early_stop':'0', 'epoch_window_cap':str(EPOCH_WINDOW_CAP),
           'amp':'true', 'amp_dtype':'float16'}
for row in rows:
    if row['parameter'] in changes:
        row['value'] = changes[row['parameter']]
local_config = project / 'transition final/parameters_25fps_active.csv'
with local_config.open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=['parameter','value','status','description'])
    w.writeheader(); w.writerows(rows)
# parameters.csv는 실행 코드가 실제 적용한 값으로 저장합니다.
with (run_dir / 'parameter_descriptions.csv').open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=['parameter','value','status','description'])
    w.writeheader(); w.writerows(rows)
print('자동 저장:', run_dir)


In [ ]:
options = ['--config',str(local_config),'--device','cuda','--microbatch',str(MICROBATCH),
           '--upstream',str(project/'pathcrf'),'--frozen-cache',str(project/'frozen_for_colab'),
           '--output',str(run_dir)]
def execute(stage, extra=()):
    command = [sys.executable,'-u','transition final/run_25fps.py',stage] + options + list(extra)
    print('실행:',stage,flush=True)
    with (run_dir/f'{stage}.log').open('a',encoding='utf-8') as log:
        process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line); log.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if status:
        raise RuntimeError(f'{stage} 실패: {run_dir/(stage+".log")}')
if RUN_BENCHMARK:
    execute('benchmark',['--bench-steps','2'])


## 학습 → 최소 검증 loss 모델 평가

25 epoch를 모두 실행한 뒤 `best.pt`로 평가합니다. `last.pt`에는 optimizer·AMP scaler·난수 상태가 들어 있습니다. 중단 후 같은 노트북과 RUN_NAME으로 실행하면 마지막 완료 epoch부터 이어집니다. epoch 중간에 끊기면 그 epoch를 다시 수행합니다.

최소 loss는 **검증 경기의 H/A CE + 가중 전환 CE + CRF NLL**입니다. 검증 loss 3회 정체 시 학습률은 절반으로 줄이고, 학습은 계속합니다. 검증 Spell F1은 기록만 하며 체크포인트 선택에 쓰지 않습니다.


In [ ]:
execute('train')
execute('test')
import pandas as pd
display(pd.read_csv(run_dir/'holdout_comparison.csv'))
print('완료:',run_dir)


## 저장된 결과

Google Drive `transition_final/runs/dfl_25fps_25epochs_minloss/`에서 받습니다.

- `parameters.csv`, `parameter_descriptions.csv`, `manifest.json`, `runtime_info.json`: 실제 설정·분할·장치·AMP
- `training_history.csv`: epoch별 검증 loss / Spell F1 / 학습 시간
- `best_validation.json`: 선택된 epoch와 최소 검증 loss
- `best.pt`: 최소 검증 loss 모델 / `last.pt`: 마지막 epoch와 재개 상태
- `holdout_comparison.csv`, `holdout.json`: 예측 D 주 결과와 정답 D 진단
- `estimated_D_primary_J03WR9_H1.csv`, `H2.csv`: 원본 프레임별 25fps 결과

원본 프레임 ID와 하프별 시간이 보존됩니다. 프레임 CSV의 H=0, A=1, D=2, 정답 결측=-1입니다. 5fps 및 5+25fps 이전 실행 폴더와 분리해 저장합니다.
